In [1]:
# ================================================================
# STAGE 3 — Model Training + Hyperparameter Grid Search
# Category: Clothing, Shoes & Jewelry
# Tối ưu: Vectorized predict, Numba SGD, NearestNeighbors cho Item-CF
# ================================================================

# ── CELL 1: INSTALL & IMPORT ─────────────────────────────────────
!pip install implicit numba -q

import json, pickle, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.neighbors import NearestNeighbors
warnings.filterwarnings('ignore')

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

COL_USER   = "reviewerID"
COL_ITEM   = "asin"
COL_RATING = "overall"
COL_DATE   = "review_date"

# ── INPUT: đổi path nếu cần (add dataset từ stage0 + stage2 output) ──
BASE = Path("/kaggle/input/notebooks/tinybullheaded/stage-0-and-2-clothing-shoes-and-jewelry")
STAGE0_DIR = BASE / "stage0_artifacts"
STAGE2_DIR = BASE / "stage2_artifacts"
STAGE3_DIR = Path("/kaggle/working/stage3_artifacts")
STAGE3_DIR.mkdir(parents=True, exist_ok=True)

print("Stage0 input :", STAGE0_DIR)
print("Stage2 input :", STAGE2_DIR)
print("Stage3 output:", STAGE3_DIR)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 704.1 kB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Stage0 input : /kaggle/input/notebooks/tinybullheaded/stage-0-and-2-clothing-shoes-and-jewelry/stage0_artifacts
Stage2 input : /kaggle/input/notebooks/tinybullheaded/stage-0-and-2-clothing-shoes-and-jewelry/stage2_artifacts
Stage3 output: /kaggle/working/stage3_artifacts


In [2]:
# ── CELL 2: LOAD ARTIFACTS ───────────────────────────────────────
train      = pd.read_parquet(STAGE0_DIR / "train.parquet")
test_raw   = pd.read_parquet(STAGE0_DIR / "test.parquet")
user_stats = pd.read_parquet(STAGE0_DIR / "user_means.parquet")
item_stats = pd.read_parquet(STAGE0_DIR / "item_means.parquet")

# Stage 2
with open(STAGE2_DIR / "stage2_artifacts.pkl", "rb") as f:
    s2 = pickle.load(f)

item2idx  = s2["item2idx"]
user2idx  = s2["user2idx"]
all_items = s2["all_items"]
all_users = s2["all_users"]
um_vec    = s2["um_vec"]          # user mean vector theo thứ tự all_users
global_mean = s2["global_mean"]

item_profiles = np.load(STAGE2_DIR / "item_profiles.npy")
user_profiles = np.load(STAGE2_DIR / "user_profiles.npy")
M_norm        = sp.load_npz(str(STAGE2_DIR / "M_norm.npz"))
implicit_mat  = sp.load_npz(str(STAGE2_DIR / "implicit_matrix.npz"))

# Mean maps
if COL_USER in user_stats.columns:
    user_mean_map = dict(zip(user_stats[COL_USER], user_stats["user_mean"]))
else:
    user_mean_map = user_stats["user_mean"].to_dict()

if COL_ITEM in item_stats.columns:
    item_mean_map = dict(zip(item_stats[COL_ITEM], item_stats["item_mean"]))
else:
    item_mean_map = item_stats["item_mean"].to_dict()

# Test warm
train_users = set(train[COL_USER].unique())
train_items = set(train[COL_ITEM].unique())
mask_warm   = test_raw[COL_USER].isin(train_users) & test_raw[COL_ITEM].isin(train_items)
test_warm   = test_raw[mask_warm].reset_index(drop=True)

print(f"train     : {len(train):,}")
print(f"test_warm : {len(test_warm):,}")
print(f"users     : {len(all_users):,}  items: {len(all_items):,}")
print(f"global_mean: {global_mean:.4f}")

train     : 6,466,384
test_warm : 1,349,731
users     : 894,092  items: 323,629
global_mean: 4.2463


In [3]:
# ── CELL 3: HELPERS ──────────────────────────────────────────────
def rmse(a, b):
    a, b = np.asarray(a, np.float32), np.asarray(b, np.float32)
    return float(np.sqrt(np.mean((a - b) ** 2)))

def mae(a, b):
    a, b = np.asarray(a, np.float32), np.asarray(b, np.float32)
    return float(np.mean(np.abs(a - b)))

def evaluate(y_true, y_pred, name="model"):
    y_true = np.asarray(y_true, np.float32)
    y_pred = np.clip(np.asarray(y_pred, np.float32), 1.0, 5.0)
    res = {"model": name, "rmse": rmse(y_true, y_pred), "mae": mae(y_true, y_pred), "per_star": {}}
    for s in [1, 2, 3, 4, 5]:
        m = y_true == s
        if m.sum() > 0:
            res["per_star"][f"star_{s}"] = {
                "n": int(m.sum()),
                "rmse": rmse(y_true[m], y_pred[m]),
                "mae":  mae(y_true[m],  y_pred[m]),
            }
    return res

def print_eval(res):
    print(f"\n{'='*55}")
    print(f"Model : {res['model']}")
    print(f"RMSE  : {res['rmse']:.6f}")
    print(f"MAE   : {res['mae']:.6f}")
    for k, v in res["per_star"].items():
        print(f"  {k}: n={v['n']:,}  RMSE={v['rmse']:.4f}  MAE={v['mae']:.4f}")
    print(f"{'='*55}")

def to_jsonable(obj):
    if isinstance(obj, dict):   return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):    return obj.tolist()
    if isinstance(obj, np.integer):    return int(obj)
    if isinstance(obj, np.floating):   return float(obj)
    return obj

def baseline_predict_vec(df, user_map, item_map, gm):
    """Vectorized baseline — không dùng iterrows."""
    u = df[COL_USER].map(user_map).fillna(gm).values.astype(np.float32)
    i = df[COL_ITEM].map(item_map).fillna(gm).values.astype(np.float32)
    return np.clip((u + i) / 2.0, 1.0, 5.0)

# Baseline
y_true_warm   = test_warm[COL_RATING].values.astype(np.float32)
y_pred_base   = baseline_predict_vec(test_warm, user_mean_map, item_mean_map, global_mean)
res_base      = evaluate(y_true_warm, y_pred_base, "baseline")
print_eval(res_base)

# Train / Val split 90/10 theo thời gian
train[COL_DATE] = pd.to_datetime(train[COL_DATE])
train_s    = train.sort_values(COL_DATE).reset_index(drop=True)
cut        = int(len(train_s) * 0.90)
train_fit  = train_s.iloc[:cut].reset_index(drop=True)
val_fit    = train_s.iloc[cut:].reset_index(drop=True)

tf_users = set(train_fit[COL_USER].unique())
tf_items = set(train_fit[COL_ITEM].unique())
val_warm = val_fit[
    val_fit[COL_USER].isin(tf_users) & val_fit[COL_ITEM].isin(tf_items)
].reset_index(drop=True)

gm_fit = float(train_fit[COL_RATING].mean())
um_fit = train_fit.groupby(COL_USER)[COL_RATING].mean().astype(np.float32)
im_fit = train_fit.groupby(COL_ITEM)[COL_RATING].mean().astype(np.float32)
um_fit_map = um_fit.to_dict()
im_fit_map = im_fit.to_dict()

y_val_true   = val_warm[COL_RATING].values.astype(np.float32)
val_base_pred = baseline_predict_vec(val_warm, um_fit_map, im_fit_map, gm_fit)

print(f"\ntrain_fit={len(train_fit):,} | val_warm={len(val_warm):,}")
print(f"Val baseline RMSE={rmse(y_val_true, val_base_pred):.5f}")


Model : baseline
RMSE  : 1.128820
MAE   : 0.879962
  star_1: n=73,663  RMSE=3.1308  MAE=3.1211
  star_2: n=77,335  RMSE=2.1588  MAE=2.1475
  star_3: n=136,137  RMSE=1.2019  MAE=1.1842
  star_4: n=235,084  RMSE=0.2966  MAE=0.2579
  star_5: n=827,512  RMSE=0.7125  MAE=0.6887

train_fit=5,819,745 | val_warm=533,477
Val baseline RMSE=1.10618


In [4]:
# ================================================================
# CELL 4: MODEL A — Item-Item CF
# Dùng sklearn NearestNeighbors + vectorized predict
# Tránh hoàn toàn iterrows và .todense() trong vòng lặp
# ================================================================
print("\n" + "="*60)
print("MODEL A — Item-Item CF (NearestNeighbors)")
print("="*60)

def build_item_cf_artifacts(df_source):
    """Build user-mean-centered rating matrix và precompute item neighbors."""
    users = sorted(df_source[COL_USER].unique())
    items = sorted(df_source[COL_ITEM].unique())
    u2i   = {u: i for i, u in enumerate(users)}
    i2i   = {it: i for i, it in enumerate(items)}

    u_idx = df_source[COL_USER].map(u2i).values.astype(np.int32)
    i_idx = df_source[COL_ITEM].map(i2i).values.astype(np.int32)
    r_arr = df_source[COL_RATING].values.astype(np.float32)

    M_raw = sp.csr_matrix((r_arr, (u_idx, i_idx)),
                           shape=(len(users), len(items)), dtype=np.float32)

    # User-mean centering (vectorized)
    user_counts = np.diff(M_raw.indptr).astype(np.int32)
    user_sums   = np.asarray(M_raw.sum(axis=1)).ravel()
    user_means  = user_sums / np.maximum(user_counts, 1)

    M_centered = M_raw.copy()
    row_idx    = np.repeat(np.arange(len(users)), np.diff(M_centered.indptr))
    M_centered.data -= user_means[row_idx]

    return {
        "u2i": u2i, "i2i": i2i,
        "M_raw": M_raw, "M_centered": M_centered,
        "user_means": user_means,
        "users": users, "items": items,
    }

def precompute_neighbors(M_centered_csr, max_k=50):
    """NearestNeighbors trên item-space — nhanh hơn nhiều so với manual cosine."""
    X_items = M_centered_csr.T.tocsr()  # shape (n_items, n_users)
    k       = min(max_k + 1, X_items.shape[0])
    nn      = NearestNeighbors(n_neighbors=k, metric="cosine",
                                algorithm="brute", n_jobs=-1)
    nn.fit(X_items)
    dist, idx = nn.kneighbors(X_items)
    sims = (1.0 - dist[:, 1:]).astype(np.float32)   # bỏ item chính nó
    idx  = idx[:, 1:].astype(np.int32)
    return idx, sims                                  # (n_items, max_k)

def predict_itemcf_vec(df_test, artifacts, nbr_idx, nbr_sim,
                        um_map, im_map, gm, top_k=20, min_co=1, shrink=5.0):
    """
    Vectorized Item-CF predict.
    Với mỗi (user, item) trong df_test:
    - Lấy neighbors của item
    - Tìm giao với những item user đã rate (dùng sparse row lookup)
    - Tính weighted sum — không iterrows, không .todense()
    """
    u2i  = artifacts["u2i"]
    i2i  = artifacts["i2i"]
    M_c  = artifacts["M_centered"].tocsr()
    umeans = artifacts["user_means"]

    n     = len(df_test)
    preds = np.empty(n, dtype=np.float32)
    fb    = 0

    # Precompute B^T @ B để đếm co-raters nhanh (1 lần duy nhất)
    B = (M_c != 0).astype(np.float32)
    BtB = (B.T @ B).tocsr()

    users_arr = df_test[COL_USER].values
    items_arr = df_test[COL_ITEM].values

    for t in range(n):
        uid, iid = users_arr[t], items_arr[t]
        base = np.clip((um_map.get(uid, gm) + im_map.get(iid, gm)) / 2, 1, 5)

        if uid not in u2i or iid not in i2i:
            preds[t] = base; fb += 1; continue

        u_l  = u2i[uid]
        i_l  = i2i[iid]
        row  = M_c.getrow(u_l)
        rated_items = row.indices
        rated_vals  = row.data

        if len(rated_items) == 0:
            preds[t] = base; fb += 1; continue

        # Neighbors của item cần predict
        nbrs_i  = nbr_idx[i_l, :top_k]
        sims_i  = nbr_sim[i_l, :top_k]

        # Co-rater counts từ BtB (O(1) lookup)
        co_counts = np.asarray(BtB.getrow(i_l)[:, nbrs_i].todense()).ravel().astype(np.int32)
        valid     = (co_counts >= min_co) & (sims_i > 0)

        if valid.sum() == 0:
            preds[t] = base; fb += 1; continue

        nbrs_v  = nbrs_i[valid]
        sims_v  = sims_i[valid] * co_counts[valid] / (co_counts[valid] + shrink)

        # Tìm giao: rated_items ∩ nbrs_v
        rated_set = dict(zip(rated_items, rated_vals))
        ws, vs    = [], []
        for j, s in zip(nbrs_v, sims_v):
            if j in rated_set:
                ws.append(s); vs.append(rated_set[j])

        if not ws:
            preds[t] = base; fb += 1; continue

        ws = np.array(ws, np.float32)
        vs = np.array(vs, np.float32)
        pred = np.clip(umeans[u_l] + np.dot(ws, vs) / (np.abs(ws).sum() + 1e-9), 1, 5)
        preds[t] = pred

    return preds, fb / n

# Build artifacts cho train_fit
print("Building item-CF artifacts (train_fit)...")
a_fit  = build_item_cf_artifacts(train_fit)
print(f"Precomputing neighbors (max_k=50)...")
nbr_idx_fit, nbr_sim_fit = precompute_neighbors(a_fit["M_centered"], max_k=50)

# Grid search trên val sample
val_s  = val_warm.sample(min(10000, len(val_warm)), random_state=42)

GRID_A = [
    dict(top_k=20, min_co=1, shrink=0.0),
    dict(top_k=20, min_co=1, shrink=2.0),
    dict(top_k=20, min_co=1, shrink=5.0),
    dict(top_k=20, min_co=2, shrink=2.0),
    dict(top_k=50, min_co=1, shrink=2.0),
    dict(top_k=50, min_co=2, shrink=5.0),
    dict(top_k=10, min_co=1, shrink=2.0),
    dict(top_k=30, min_co=1, shrink=5.0),
]
print(f"\nGrid search {len(GRID_A)} configs on {len(val_s)} samples...")

best_A_rmse, best_A_params, res_A = np.inf, None, []
for p in GRID_A:
    pv, fb = predict_itemcf_vec(val_s, a_fit, nbr_idx_fit, nbr_sim_fit,
                                 um_fit_map, im_fit_map, gm_fit, **p)
    r, m = rmse(val_s[COL_RATING].values, pv), mae(val_s[COL_RATING].values, pv)
    res_A.append({**p, "val_rmse": r, "val_mae": m, "fallback": fb})
    print(f"  {p} -> RMSE={r:.5f} MAE={m:.5f} fb={fb:.3f}")
    if r < best_A_rmse:
        best_A_rmse, best_A_params = r, p

print(f"\nBest A params: {best_A_params}")

# OOF predictions trên toàn val_warm
print("Computing OOF preds (val_warm)...")
model_a_val_preds_oof, _ = predict_itemcf_vec(
    val_warm, a_fit, nbr_idx_fit, nbr_sim_fit,
    um_fit_map, im_fit_map, gm_fit, **best_A_params
)
res_a_oof = evaluate(y_val_true, model_a_val_preds_oof, "model_a_val_oof")
print_eval(res_a_oof)

# Refit trên full train
print("\nRefitting Model A on full train...")
a_full = build_item_cf_artifacts(train)
nbr_idx_full, nbr_sim_full = precompute_neighbors(a_full["M_centered"], max_k=50)

model_a_preds, fb_a = predict_itemcf_vec(
    test_warm, a_full, nbr_idx_full, nbr_sim_full,
    user_mean_map, item_mean_map, global_mean, **best_A_params
)
res_a = evaluate(y_true_warm, model_a_preds, "model_a_full")
print_eval(res_a)
print(f"Fallback rate: {fb_a:.3f}")

np.save(STAGE3_DIR / "model_a_val_preds_oof.npy", model_a_val_preds_oof)
np.save(STAGE3_DIR / "model_a_preds.npy", model_a_preds)
pd.DataFrame(res_A).to_csv(STAGE3_DIR / "model_a_tuning.csv", index=False)
with open(STAGE3_DIR / "model_a_result.json", "w") as f:
    json.dump(to_jsonable(res_a), f, indent=2)
print("Model A saved.")


MODEL A — Item-Item CF (NearestNeighbors)
Building item-CF artifacts (train_fit)...
Precomputing neighbors (max_k=50)...

Grid search 8 configs on 10000 samples...
  {'top_k': 20, 'min_co': 1, 'shrink': 0.0} -> RMSE=1.10502 MAE=0.83692 fb=0.995
  {'top_k': 20, 'min_co': 1, 'shrink': 2.0} -> RMSE=1.10502 MAE=0.83692 fb=0.995
  {'top_k': 20, 'min_co': 1, 'shrink': 5.0} -> RMSE=1.10502 MAE=0.83692 fb=0.995
  {'top_k': 20, 'min_co': 2, 'shrink': 2.0} -> RMSE=1.10529 MAE=0.83737 fb=0.998
  {'top_k': 50, 'min_co': 1, 'shrink': 2.0} -> RMSE=1.10731 MAE=0.83735 fb=0.987
  {'top_k': 50, 'min_co': 2, 'shrink': 5.0} -> RMSE=1.10543 MAE=0.83738 fb=0.996
  {'top_k': 10, 'min_co': 1, 'shrink': 2.0} -> RMSE=1.10490 MAE=0.83713 fb=0.997
  {'top_k': 30, 'min_co': 1, 'shrink': 5.0} -> RMSE=1.10598 MAE=0.83698 fb=0.992

Best A params: {'top_k': 10, 'min_co': 1, 'shrink': 2.0}
Computing OOF preds (val_warm)...

Model : model_a_val_oof
RMSE  : 1.106776
MAE   : 0.832440
  star_1: n=26,108  RMSE=3.0091  MAE

In [5]:
# ================================================================
# CELL 5: MODEL B — Matrix Factorization SGD (Numba JIT)
# ================================================================
print("\n" + "="*60)
print("MODEL B — Matrix Factorization SGD (Numba)")
print("="*60)

try:
    from numba import njit
    USE_NUMBA = True
    print("Numba OK")
except ImportError:
    USE_NUMBA = False
    print("Numba not found — dùng NumPy fallback")

if USE_NUMBA:
    @njit(cache=True)
    def _sgd_epoch(u_idx, i_idx, ratings, P, Q, bu, bi, gm, lr, reg):
        n = len(ratings)
        for k in range(n):
            u, i, r = u_idx[k], i_idx[k], ratings[k]
            pred = gm + bu[u] + bi[i]
            for f in range(P.shape[1]):
                pred += P[u, f] * Q[i, f]
            err = r - pred
            bu[u] += lr * (err - reg * bu[u])
            bi[i] += lr * (err - reg * bi[i])
            for f in range(P.shape[1]):
                pu = P[u, f]
                P[u, f] += lr * (err * Q[i, f] - reg * pu)
                Q[i, f] += lr * (err * pu        - reg * Q[i, f])
        return P, Q, bu, bi
else:
    def _sgd_epoch(u_idx, i_idx, ratings, P, Q, bu, bi, gm, lr, reg):
        for k in range(len(ratings)):
            u, i, r = u_idx[k], i_idx[k], float(ratings[k])
            err = r - (gm + bu[u] + bi[i] + P[u].dot(Q[i]))
            bu[u] += lr * (err - reg * bu[u])
            bi[i] += lr * (err - reg * bi[i])
            pu = P[u].copy()
            P[u] += lr * (err * Q[i] - reg * P[u])
            Q[i] += lr * (err * pu   - reg * Q[i])
        return P, Q, bu, bi

def train_mf(df, n_factors, n_epochs, lr, reg, gm, lr_decay=0.95, seed=42, verbose=True):
    np.random.seed(seed)
    users = sorted(df[COL_USER].unique())
    items = sorted(df[COL_ITEM].unique())
    u2i   = {u: i for i, u in enumerate(users)}
    i2i   = {it: i for i, it in enumerate(items)}

    ui = df[COL_USER].map(u2i).values.astype(np.int32)
    ii = df[COL_ITEM].map(i2i).values.astype(np.int32)
    rr = df[COL_RATING].values.astype(np.float64)

    P  = np.random.normal(0, 0.1, (len(users), n_factors))
    Q  = np.random.normal(0, 0.1, (len(items), n_factors))
    bu = np.zeros(len(users))
    bi = np.zeros(len(items))

    history = []
    cur_lr  = float(lr)
    for ep in range(n_epochs):
        perm = np.random.permutation(len(rr))
        P, Q, bu, bi = _sgd_epoch(
            ui[perm], ii[perm], rr[perm].astype(np.float64),
            P, Q, bu, bi, gm, cur_lr, reg
        )
        cur_lr *= lr_decay
        p_tr   = gm + bu[ui] + bi[ii] + (P[ui] * Q[ii]).sum(1)
        ep_rmse = float(np.sqrt(np.mean((rr - p_tr) ** 2)))
        history.append({"epoch": ep + 1, "train_rmse": ep_rmse})
        if verbose:
            print(f"  ep{ep+1:02d}: train RMSE={ep_rmse:.5f}")

    return {"P": P, "Q": Q, "bu": bu, "bi": bi,
            "u2i": u2i, "i2i": i2i, "gm": gm, "history": history}

def predict_mf_vec(df, model, um_map, im_map, gm):
    """Vectorized MF predict — không dùng iterrows."""
    P, Q, bu, bi = model["P"], model["Q"], model["bu"], model["bi"]
    u2i, i2i     = model["u2i"], model["i2i"]

    u_idx = df[COL_USER].map(u2i).values        # NaN nếu unknown
    i_idx = df[COL_ITEM].map(i2i).values

    # Baseline fallback
    u_base = df[COL_USER].map(um_map).fillna(gm).values.astype(np.float32)
    i_base = df[COL_ITEM].map(im_map).fillna(gm).values.astype(np.float32)
    preds  = np.clip((u_base + i_base) / 2.0, 1.0, 5.0)

    # Known mask
    known = pd.notna(u_idx) & pd.notna(i_idx)
    if known.any():
        uk = u_idx[known].astype(int)
        ik = i_idx[known].astype(int)
        mf_pred = gm + bu[uk] + bi[ik] + (P[uk] * Q[ik]).sum(axis=1)
        preds[known] = np.clip(mf_pred, 1.0, 5.0).astype(np.float32)

    return preds

GRID_B = [
    dict(n_factors=20,  n_epochs=5,  lr=0.01,  reg=0.02),
    dict(n_factors=20,  n_epochs=10, lr=0.01,  reg=0.02),
    dict(n_factors=50,  n_epochs=5,  lr=0.01,  reg=0.02),
    dict(n_factors=50,  n_epochs=10, lr=0.01,  reg=0.02),
    dict(n_factors=50,  n_epochs=5,  lr=0.02,  reg=0.02),
    dict(n_factors=50,  n_epochs=10, lr=0.02,  reg=0.05),
    dict(n_factors=100, n_epochs=5,  lr=0.01,  reg=0.02),
    dict(n_factors=100, n_epochs=10, lr=0.01,  reg=0.05),
    dict(n_factors=20,  n_epochs=20, lr=0.005, reg=0.02),
    dict(n_factors=50,  n_epochs=20, lr=0.005, reg=0.02),
]
print(f"Grid search {len(GRID_B)} configs...")

best_B_rmse, best_B_params, best_B_model, res_B = np.inf, None, None, []
for p in GRID_B:
    print(f"\n  Config: {p}")
    m = train_mf(train_fit, **p, gm=gm_fit, verbose=True)
    pv = predict_mf_vec(val_warm, m, um_fit_map, im_fit_map, gm_fit)
    r, mv = rmse(y_val_true, pv), mae(y_val_true, pv)
    res_B.append({**p, "val_rmse": r, "val_mae": mv})
    print(f"  => Val RMSE={r:.5f}  MAE={mv:.5f}")
    if r < best_B_rmse:
        best_B_rmse, best_B_params, best_B_model = r, p, m

print(f"\nBest B: {best_B_params}")

# OOF
model_b_val_preds_oof = predict_mf_vec(val_warm, best_B_model, um_fit_map, im_fit_map, gm_fit)
res_b_oof = evaluate(y_val_true, model_b_val_preds_oof, "model_b_val_oof")
print_eval(res_b_oof)

# Refit on full train
print("\nRefitting Model B on full train...")
model_B_full = train_mf(train, **best_B_params, gm=global_mean, verbose=True)
model_b_preds = predict_mf_vec(test_warm, model_B_full, user_mean_map, item_mean_map, global_mean)
res_b = evaluate(y_true_warm, model_b_preds, "model_b_full")
print_eval(res_b)

# Per-star
for s in [1, 2, 3, 4, 5]:
    mask = y_true_warm == s
    if mask.sum() == 0: continue
    print(f"  {s}★ n={mask.sum():,} RMSE={rmse(y_true_warm[mask], model_b_preds[mask]):.4f} "
          f"MAE={mae(y_true_warm[mask], model_b_preds[mask]):.4f}")

np.save(STAGE3_DIR / "model_b_val_preds_oof.npy", model_b_val_preds_oof)
np.save(STAGE3_DIR / "model_b_preds.npy", model_b_preds)
np.save(STAGE3_DIR / "model_b_P.npy", model_B_full["P"])
np.save(STAGE3_DIR / "model_b_Q.npy", model_B_full["Q"])
pd.DataFrame(res_B).to_csv(STAGE3_DIR / "model_b_tuning.csv", index=False)
with open(STAGE3_DIR / "model_b_result.json", "w") as f:
    json.dump(to_jsonable(res_b), f, indent=2)
print("Model B saved.")


MODEL B — Matrix Factorization SGD (Numba)
Numba OK
Grid search 10 configs...

  Config: {'n_factors': 20, 'n_epochs': 5, 'lr': 0.01, 'reg': 0.02}
  ep01: train RMSE=1.07326
  ep02: train RMSE=1.04030
  ep03: train RMSE=1.01602
  ep04: train RMSE=0.99664
  ep05: train RMSE=0.98042
  => Val RMSE=1.09068  MAE=0.84342

  Config: {'n_factors': 20, 'n_epochs': 10, 'lr': 0.01, 'reg': 0.02}
  ep01: train RMSE=1.07326
  ep02: train RMSE=1.04030
  ep03: train RMSE=1.01602
  ep04: train RMSE=0.99664
  ep05: train RMSE=0.98042
  ep06: train RMSE=0.96640
  ep07: train RMSE=0.95393
  ep08: train RMSE=0.94261
  ep09: train RMSE=0.93215
  ep10: train RMSE=0.92237
  => Val RMSE=1.08447  MAE=0.82858

  Config: {'n_factors': 50, 'n_epochs': 5, 'lr': 0.01, 'reg': 0.02}
  ep01: train RMSE=1.06802
  ep02: train RMSE=1.02956
  ep03: train RMSE=1.00018
  ep04: train RMSE=0.97587
  ep05: train RMSE=0.95475
  => Val RMSE=1.09229  MAE=0.84442

  Config: {'n_factors': 50, 'n_epochs': 10, 'lr': 0.01, 'reg': 0.02

In [6]:
# ================================================================
# CELL 6: MODEL C — Content-Based Filtering (Vectorized)
# ================================================================
print("\n" + "="*60)
print("MODEL C — Content-Based Filtering")
print("="*60)

def l2_norm_rows(X):
    X = np.asarray(X, np.float32)
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)

def build_content_artifacts(df_source, item_profiles_norm, item_enc):
    """
    Xây dựng user profiles từ df_source bằng sparse matmul.
    Hoàn toàn vectorized — không iterrows.
    """
    items_in_enc = [iid for iid in df_source[COL_ITEM].unique() if iid in item_enc]
    item_enc_local = {iid: i for i, iid in enumerate(items_in_enc)}

    df_k = df_source[df_source[COL_ITEM].isin(item_enc_local)].copy()
    users = sorted(df_k[COL_USER].unique())
    user_enc_local = {uid: i for i, uid in enumerate(users)}

    u_idx = df_k[COL_USER].map(user_enc_local).values.astype(np.int32)
    i_idx = df_k[COL_ITEM].map(item_enc_local).values.astype(np.int32)

    # Item profile rows theo thứ tự items_in_enc
    global_item_rows = np.array([item_enc[iid] for iid in items_in_enc], dtype=np.int32)
    ip_local = item_profiles_norm[global_item_rows]  # (n_items_local, 128)

    # Sparse user-item matrix (binary) → user profiles = avg item profiles
    vals   = np.ones(len(df_k), dtype=np.float32)
    UI_csr = sp.csr_matrix((vals, (u_idx, i_idx)),
                            shape=(len(users), len(items_in_enc)), dtype=np.float32)
    counts   = np.asarray(UI_csr.sum(axis=1)).ravel()
    up_local = (UI_csr @ ip_local) / np.maximum(counts[:, None], 1.0)
    up_local = l2_norm_rows(up_local)

    return {
        "user_enc": user_enc_local,
        "item_enc": item_enc_local,
        "user_profiles": up_local.astype(np.float32),
        "item_profiles": ip_local.astype(np.float32),
    }

def predict_content_vec(df, ca, baseline_arr, residual_scale):
    """Vectorized content-based predict — dùng matrix multiply."""
    preds = baseline_arr.copy().astype(np.float32)

    known = df[COL_USER].isin(ca["user_enc"]) & df[COL_ITEM].isin(ca["item_enc"])
    if known.any():
        df_k  = df[known]
        u_idx = df_k[COL_USER].map(ca["user_enc"]).values.astype(np.int32)
        i_idx = df_k[COL_ITEM].map(ca["item_enc"]).values.astype(np.int32)
        # Cosine sim = dot product (cả hai đã L2-normalize)
        cos_sim = (ca["user_profiles"][u_idx] * ca["item_profiles"][i_idx]).sum(axis=1)
        preds[known.values] = np.clip(
            baseline_arr[known.values] + float(residual_scale) * cos_sim, 1.0, 5.0
        )
    return preds

# Build content artifacts
ip_norm = l2_norm_rows(item_profiles)
print("Building content artifacts (train_fit)...")
ca_fit = build_content_artifacts(train_fit, ip_norm, item2idx)

SCALES_C = [0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.75, 1.0, 1.5, 2.0]
print(f"Searching residual_scale: {SCALES_C}")

best_C_rmse, best_C_scale, res_C = np.inf, 0.0, []
for sc in SCALES_C:
    pv = predict_content_vec(val_warm, ca_fit, val_base_pred, sc)
    r, m = rmse(y_val_true, pv), mae(y_val_true, pv)
    res_C.append({"scale": sc, "val_rmse": r, "val_mae": m})
    print(f"  scale={sc:.2f} -> RMSE={r:.5f}  MAE={m:.5f}")
    if r < best_C_rmse:
        best_C_rmse, best_C_scale = r, sc

print(f"\nBest C scale: {best_C_scale}")

# OOF
model_c_val_preds_oof = predict_content_vec(val_warm, ca_fit, val_base_pred, best_C_scale)
res_c_oof = evaluate(y_val_true, model_c_val_preds_oof, "model_c_val_oof")
print_eval(res_c_oof)

# Refit on full train
print("\nBuilding content artifacts (full train)...")
ca_full = build_content_artifacts(train, ip_norm, item2idx)
model_c_preds = predict_content_vec(test_warm, ca_full, y_pred_base, best_C_scale)
res_c = evaluate(y_true_warm, model_c_preds, "model_c_full")
print_eval(res_c)

np.save(STAGE3_DIR / "model_c_val_preds_oof.npy", model_c_val_preds_oof)
np.save(STAGE3_DIR / "model_c_preds.npy", model_c_preds)
pd.DataFrame(res_C).to_csv(STAGE3_DIR / "model_c_tuning.csv", index=False)
with open(STAGE3_DIR / "model_c_result.json", "w") as f:
    json.dump(to_jsonable(res_c), f, indent=2)
print("Model C saved.")


MODEL C — Content-Based Filtering
Building content artifacts (train_fit)...
Searching residual_scale: [0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.75, 1.0, 1.5, 2.0]
  scale=0.00 -> RMSE=1.10618  MAE=0.83283
  scale=0.05 -> RMSE=1.10676  MAE=0.81966
  scale=0.10 -> RMSE=1.10880  MAE=0.80698
  scale=0.15 -> RMSE=1.11223  MAE=0.79490
  scale=0.20 -> RMSE=1.11697  MAE=0.78352
  scale=0.30 -> RMSE=1.12989  MAE=0.76339
  scale=0.50 -> RMSE=1.16510  MAE=0.73563
  scale=0.75 -> RMSE=1.21514  MAE=0.72124
  scale=1.00 -> RMSE=1.25979  MAE=0.72017
  scale=1.50 -> RMSE=1.31592  MAE=0.72729
  scale=2.00 -> RMSE=1.33991  MAE=0.73187

Best C scale: 0.0

Model : model_c_val_oof
RMSE  : 1.106182
MAE   : 0.832835
  star_1: n=26,108  RMSE=3.0093  MAE=2.9576
  star_2: n=28,860  RMSE=2.0879  MAE=2.0303
  star_3: n=52,755  RMSE=1.1990  MAE=1.1218
  star_4: n=95,555  RMSE=0.4671  MAE=0.3823
  star_5: n=330,199  RMSE=0.7662  MAE=0.6444

Building content artifacts (full train)...

Model : model_c_full
RMSE  : 1.1

In [7]:
# ================================================================
# CELL 7: MODEL D — ALS Implicit (vectorized predict)
# ================================================================
print("\n" + "="*60)
print("MODEL D — ALS Implicit Feedback")
print("="*60)

try:
    import implicit
    ALS_OK = True
    print(f"implicit {implicit.__version__}")
except ImportError:
    ALS_OK = False
    print("implicit not available — fallback to baseline")

def build_implicit_csr(df_source):
    users = sorted(df_source[COL_USER].unique())
    items = sorted(df_source[COL_ITEM].unique())
    u2i   = {u: i for i, u in enumerate(users)}
    i2i   = {it: i for i, it in enumerate(items)}
    u_idx = df_source[COL_USER].map(u2i).values.astype(np.int32)
    i_idx = df_source[COL_ITEM].map(i2i).values.astype(np.int32)
    C = sp.csr_matrix(
        (np.ones(len(df_source), dtype=np.float32), (u_idx, i_idx)),
        shape=(len(users), len(items))
    )
    C.sum_duplicates()
    return C, u2i, i2i

def predict_als_vec(df, U, V, u2i, i2i, baseline_arr, residual_scale):
    """Vectorized ALS predict — không iterrows."""
    preds = baseline_arr.copy().astype(np.float32)
    u_idx = df[COL_USER].map(u2i).values
    i_idx = df[COL_ITEM].map(i2i).values
    known = pd.notna(u_idx) & pd.notna(i_idx)
    if known.any():
        uk = u_idx[known].astype(int)
        ik = i_idx[known].astype(int)
        scores = (U[uk] * V[ik]).sum(axis=1).astype(np.float32)
        preds[known] = np.clip(baseline_arr[known] + residual_scale * scores, 1.0, 5.0)
    return preds

# Build implicit matrix cho train_fit
C_fit, u2i_fit, i2i_fit = build_implicit_csr(train_fit)
val_base_d = val_base_pred.copy()

GRID_D = [
    dict(factors=20,  regularization=0.005, iterations=10),
    dict(factors=20,  regularization=0.005, iterations=15),
    dict(factors=20,  regularization=0.01,  iterations=15),
    dict(factors=50,  regularization=0.005, iterations=10),
    dict(factors=50,  regularization=0.01,  iterations=15),
    dict(factors=100, regularization=0.01,  iterations=10),
]
SCALE_GRID_D = np.round(np.linspace(0.8, 1.5, 15), 3).tolist() + [0.0, 0.1, 0.3, 0.5]
SCALE_GRID_D = sorted(set(SCALE_GRID_D))

print(f"Grid search {len(GRID_D)} ALS configs × {len(SCALE_GRID_D)} scales...")

best_D_rmse, best_D_params, res_D = np.inf, None, []
best_U_d, best_V_d = None, None
best_u2i_d, best_i2i_d = None, None
best_rs_d = 0.0
model_d_val_preds_oof = val_base_pred.copy()

if ALS_OK:
    for p in GRID_D:
        als = implicit.als.AlternatingLeastSquares(
            factors=p["factors"], regularization=p["regularization"],
            iterations=p["iterations"], use_gpu=False, random_state=42
        )
        als.fit(C_fit)
        U_tmp, V_tmp = als.user_factors, als.item_factors

        # Tìm best scale cho config này
        best_rs_local, best_r_local = 0.0, np.inf
        for rs in SCALE_GRID_D:
            pv = predict_als_vec(val_warm, U_tmp, V_tmp, u2i_fit, i2i_fit,
                                  val_base_d, rs)
            r  = rmse(y_val_true, pv)
            if r < best_r_local:
                best_r_local, best_rs_local = r, rs
        m_local = mae(y_val_true, predict_als_vec(
            val_warm, U_tmp, V_tmp, u2i_fit, i2i_fit, val_base_d, best_rs_local))

        res_D.append({**p, "residual_scale": best_rs_local,
                      "val_rmse": best_r_local, "val_mae": m_local})
        print(f"  {p} | best_rs={best_rs_local} -> RMSE={best_r_local:.5f} MAE={m_local:.5f}")

        if best_r_local < best_D_rmse:
            best_D_rmse  = best_r_local
            best_D_params = {**p, "residual_scale": best_rs_local}
            best_U_d, best_V_d = U_tmp.copy(), V_tmp.copy()
            best_u2i_d, best_i2i_d = u2i_fit, i2i_fit
            best_rs_d = best_rs_local

    model_d_val_preds_oof = predict_als_vec(
        val_warm, best_U_d, best_V_d, best_u2i_d, best_i2i_d,
        val_base_d, best_rs_d
    )

res_d_oof = evaluate(y_val_true, model_d_val_preds_oof, "model_d_val_oof")
print_eval(res_d_oof)

# Refit on full train
model_d_preds = y_pred_base.copy()
if ALS_OK and best_D_params:
    print(f"\nRefitting Model D on full train: {best_D_params}")
    C_full, u2i_full, i2i_full = build_implicit_csr(train)
    als_f = implicit.als.AlternatingLeastSquares(
        factors=best_D_params["factors"],
        regularization=best_D_params["regularization"],
        iterations=best_D_params["iterations"],
        use_gpu=False, random_state=42
    )
    als_f.fit(C_full)
    model_d_preds = predict_als_vec(
        test_warm, als_f.user_factors, als_f.item_factors,
        u2i_full, i2i_full, y_pred_base, best_D_params["residual_scale"]
    )

res_d = evaluate(y_true_warm, model_d_preds, "model_d_full")
print_eval(res_d)

np.save(STAGE3_DIR / "model_d_val_preds_oof.npy", model_d_val_preds_oof)
np.save(STAGE3_DIR / "model_d_preds.npy", model_d_preds)
pd.DataFrame(res_D).to_csv(STAGE3_DIR / "model_d_tuning.csv", index=False)
with open(STAGE3_DIR / "model_d_result.json", "w") as f:
    json.dump(to_jsonable(res_d), f, indent=2)
print("Model D saved.")


MODEL D — ALS Implicit Feedback
implicit 0.7.2
Grid search 6 ALS configs × 19 scales...


  0%|          | 0/10 [00:00<?, ?it/s]

  {'factors': 20, 'regularization': 0.005, 'iterations': 10} | best_rs=0.3 -> RMSE=1.10618 MAE=0.83281


  0%|          | 0/15 [00:00<?, ?it/s]

  {'factors': 20, 'regularization': 0.005, 'iterations': 15} | best_rs=0.3 -> RMSE=1.10618 MAE=0.83281


...

  {'factors': 50, 'regularization': 0.01, 'iterations': 15} | best_rs=0.1 -> RMSE=1.10618 MAE=0.83282


  0%|          | 0/10 [00:00<?, ?it/s]

  {'factors': 100, 'regularization': 0.01, 'iterations': 10} | best_rs=0.1 -> RMSE=1.10618 MAE=0.83282

Model : model_d_val_oof
RMSE  : 1.106180
MAE   : 0.832812
  star_1: n=26,108  RMSE=3.0093  MAE=2.9576
  star_2: n=28,860  RMSE=2.0879  MAE=2.0304
  star_3: n=52,755  RMSE=1.1991  MAE=1.1218
  star_4: n=95,555  RMSE=0.4672  MAE=0.3823
  star_5: n=330,199  RMSE=0.7662  MAE=0.6443

Refitting Model D on full train: {'factors': 20, 'regularization': 0.005, 'iterations': 15, 'residual_scale': 0.3}


  0%|          | 0/15 [00:00<?, ?it/s]


Model : model_d_full
RMSE  : 1.128820
MAE   : 0.879938
  star_1: n=73,663  RMSE=3.1308  MAE=3.1212
  star_2: n=77,335  RMSE=2.1589  MAE=2.1476
  star_3: n=136,137  RMSE=1.2019  MAE=1.1842
  star_4: n=235,084  RMSE=0.2967  MAE=0.2580
  star_5: n=827,512  RMSE=0.7124  MAE=0.6886
Model D saved.


In [8]:
# ================================================================
# CELL 8: TỔNG KẾT + EXPORT OOF CHO STAGE 4
# ================================================================
print("\n" + "="*70)
print("STAGE 3 SUMMARY — Clothing, Shoes & Jewelry")
print("="*70)
print(f"{'Model':<25} {'Test RMSE':>10} {'Test MAE':>10} {'ΔRMSE':>10}")
print("-"*60)

base_r = res_base["rmse"]
for name, preds in [
    ("Baseline",          y_pred_base),
    ("Model A (Item-CF)", model_a_preds),
    ("Model B (MF-SGD)",  model_b_preds),
    ("Model C (Content)", model_c_preds),
    ("Model D (ALS)",     model_d_preds),
]:
    r = rmse(y_true_warm, preds)
    m = mae(y_true_warm,  preds)
    d = (r - base_r) / base_r * 100
    print(f"{name:<25} {r:>10.5f} {m:>10.5f} {d:>+9.3f}%")

# Sanity check
for name, arr in [
    ("model_a_val", model_a_val_preds_oof),
    ("model_b_val", model_b_val_preds_oof),
    ("model_c_val", model_c_val_preds_oof),
    ("model_d_val", model_d_val_preds_oof),
]:
    assert len(arr) == len(val_warm), f"OOF length mismatch: {name}"
    assert np.isfinite(arr).all(),    f"Non-finite in {name}"

# Export OOF cho Stage 4
oof_df = pd.DataFrame({
    COL_USER:   val_warm[COL_USER].values,
    COL_ITEM:   val_warm[COL_ITEM].values,
    COL_RATING: y_val_true,
    "baseline": val_base_pred,
    "model_a":  model_a_val_preds_oof,
    "model_b":  model_b_val_preds_oof,
    "model_c":  model_c_val_preds_oof,
    "model_d":  model_d_val_preds_oof,
})

test_df = pd.DataFrame({
    COL_USER:   test_warm[COL_USER].values,
    COL_ITEM:   test_warm[COL_ITEM].values,
    COL_RATING: y_true_warm,
    "baseline": y_pred_base,
    "model_a":  model_a_preds,
    "model_b":  model_b_preds,
    "model_c":  model_c_preds,
    "model_d":  model_d_preds,
})

oof_df.to_parquet(STAGE3_DIR / "stage3_oof.parquet", index=False)
test_df.to_parquet(STAGE3_DIR / "stage3_test_preds.parquet", index=False)

# Summary JSON
summary = {
    "best_configs": {
        "model_a": best_A_params,
        "model_b": best_B_params,
        "model_c": {"residual_scale": float(best_C_scale)},
        "model_d": to_jsonable(best_D_params) if best_D_params else None,
    },
    "val_metrics": {
        "baseline": {"rmse": rmse(y_val_true, val_base_pred), "mae": mae(y_val_true, val_base_pred)},
        "model_a":  {"rmse": res_a_oof["rmse"], "mae": res_a_oof["mae"]},
        "model_b":  {"rmse": res_b_oof["rmse"], "mae": res_b_oof["mae"]},
        "model_c":  {"rmse": res_c_oof["rmse"], "mae": res_c_oof["mae"]},
        "model_d":  {"rmse": res_d_oof["rmse"], "mae": res_d_oof["mae"]},
    },
    "test_metrics": {
        "baseline": {"rmse": res_base["rmse"], "mae": res_base["mae"]},
        "model_a":  {"rmse": res_a["rmse"],    "mae": res_a["mae"]},
        "model_b":  {"rmse": res_b["rmse"],    "mae": res_b["mae"]},
        "model_c":  {"rmse": res_c["rmse"],    "mae": res_c["mae"]},
        "model_d":  {"rmse": res_d["rmse"],    "mae": res_d["mae"]},
    },
}
with open(STAGE3_DIR / "stage3_summary.json", "w") as f:
    json.dump(to_jsonable(summary), f, indent=2)

print(f"\nSaved to {STAGE3_DIR}")
for p in sorted(STAGE3_DIR.iterdir()):
    print(f"  {p.name:<40} {p.stat().st_size/1e6:.1f} MB")
print("\n=== STAGE 3 DONE ===")


STAGE 3 SUMMARY — Clothing, Shoes & Jewelry
Model                      Test RMSE   Test MAE      ΔRMSE
------------------------------------------------------------
Baseline                     1.12882    0.87996    +0.000%
Model A (Item-CF)            1.12938    0.87960    +0.050%
Model B (MF-SGD)             1.12059    0.85073    -0.729%
Model C (Content)            1.12882    0.87996    +0.000%
Model D (ALS)                1.12882    0.87994    +0.000%

Saved to /kaggle/working/stage3_artifacts
  model_a_preds.npy                        5.4 MB
  model_a_result.json                      0.0 MB
  model_a_tuning.csv                       0.0 MB
  model_a_val_preds_oof.npy                2.1 MB
  model_b_P.npy                            143.1 MB
  model_b_Q.npy                            51.8 MB
  model_b_preds.npy                        5.4 MB
  model_b_result.json                      0.0 MB
  model_b_tuning.csv                       0.0 MB
  model_b_val_preds_oof.npy                2